# Compare Auction Values

In [ ]:
import pandas as pd
import seaborn as sns
pd.options.display.precision = 2
import matplotlib.pyplot as plt
import json
import sys

In [ ]:
sys.path.append('../')

In [ ]:
League_path = '/Users/cmartin/Fantasy_Baseball/Ottoneu_Baseball_Projects/Imaginary_Hammers/'

In [ ]:
Calc_date = 'Mar21_2026'

In [ ]:
Full_Merge_path = League_path+'/Full_Merge.csv'

In [ ]:
TeamID = 240

In [ ]:
Replacement_Level_Recalc_path = League_path+'/Recalc_Replacement_Level.csv'

In [ ]:
Repl_Level_H_path = League_path+'/Latest_Hitter_Repl.csv'
Repl_Level_P_path = League_path+'/Latest_Pitcher_Repl.csv'

In [ ]:
Target_Stats_path = League_path+'/Target_Stats_dict.json'

In [ ]:
Additional_targets = {
    'G_mySGP':162.*12.,
    'G_FGAV': 162.*12.,
    'IP_mySGP':1500.,
    'IP_FGAV':1500.,
    'TOTAL_SGP_Val_mySGP':500., #terget value for full team
    'CAT_SGP_Val':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'Salary': 380. #assume 10 for in-season pickups
}

In [ ]:
# Val_per_SGP = {
#     'Hitter': 6.349899134101533,
#     'Pitcher': 4.757117079794198
# }

In [ ]:
Replacement_Level_Recalc_df = pd.read_csv(Replacement_Level_Recalc_path)

In [ ]:
Replacement_Level_Orig_df = pd.concat([
    pd.read_csv(Repl_Level_H_path),
    pd.read_csv(Repl_Level_P_path)
])

In [ ]:
with open(Target_Stats_path, 'r') as f:
    data = json.load(f)
    data.update(Additional_targets)
Target_Stats_df = pd.json_normalize(data)

In [ ]:
Target_Stats_df.head()

In [ ]:
Target_Stats_Hitter_df = Target_Stats_df[['Target Pts','R','HR','OBP','SLG','G_mySGP','G_FGAV','TOTAL_SGP_Val_mySGP','CAT_SGP_Val','Salary']]
Target_Stats_Hitter_df['Hitter Pitcher'] = 'Hitter'
Target_Stats_Pitcher_df = Target_Stats_df[['Target Pts','K','HR9','ERA','WHIP','IP_mySGP','IP_FGAV','TOTAL_SGP_Val_mySGP','CAT_SGP_Val','Salary']]
Target_Stats_Pitcher_df['Hitter Pitcher'] = 'Pitcher'
Target_Stats_Pitcher_df.rename(columns={'K':'SO'},inplace=True)
Target_Stats_df = pd.concat([
    Target_Stats_Hitter_df,
    Target_Stats_Pitcher_df
])

In [ ]:
Target_Stats_df.head()

In [ ]:
Player_id_cols = [
    'FG ID','Name','Team','Ottoneu ID','Ottoneu Positions'
]

In [ ]:
Fantasy_Team_ID_cols = [
    'TeamID', 'Team Name',
]

In [ ]:
Games_IP_Targets = {
    'C':162,
    '1B':162,
    '2B':162,
    '3B':162,
    'SS':162,
    'MI':162, 
    'OF':810, #total
    'Util':162, 
    'IP':1500 #total
}
Start_Spots = {
    'C':1, #technically 2 but 162 G limit
    '1B':1,
    '2B':1,
    'SS':1,
    'MI':1,
    '3B':1,
    'OF':5,
    'Util':1, #starting Hitters
    'SP':5,
    'RP':5
}

Total_Roster_Spots = {
    'C':2,
    '1B':2,
    '2B':3,
    'SS':3,
    '3B':2,
    'OF':9,
    'SP':9,
    'RP':6
}

In [ ]:
#Scoring Categories
Count_Scoring_Categories_Batting = [
    'R',
    'HR'
]
Rate_Scoring_Categories_Batting = [
    'OBP',
    'SLG'
]
Count_Scoring_Categories_Pitching = [
    'SO'
]
Rate_Scoring_Categories_Pitching = [
    "HR9",
    "ERA",
    "WHIP"
]
Num_teams = 12.
Team_budget = 400.
Hitter_sal_split = 0.53
League_budget = Team_budget*Num_teams
Hitter_budget = League_budget*Hitter_sal_split
Pitcher_budget = League_budget*(1.-Hitter_sal_split)
Scoring_Categories_Batting = Count_Scoring_Categories_Batting + Rate_Scoring_Categories_Batting
Scoring_Categories_Pitching = Count_Scoring_Categories_Pitching + Rate_Scoring_Categories_Pitching
Scoring_Categories = Scoring_Categories_Batting + Scoring_Categories_Pitching

In [ ]:
Hitting_Pos = [
    'C',
    '1B',
    '2B',
    'SS',
    '3B',
    'OF',
    'MI',
    'Util'
]
Pitching_Pos = [
    'SP',
    'RP'
]

All_Pos = Hitting_Pos + Pitching_Pos

In [ ]:
Full_Merge_df = pd.read_csv(Full_Merge_path)
Full_Merge_df['FG ID'] = Full_Merge_df['FG ID'].astype(str)
Full_Merge_df['FG ID'] = Full_Merge_df['FG ID'].str.replace('.0','')
Full_Merge_df['Ottoneu ID'] = Full_Merge_df['Ottoneu ID'].astype(str)
Full_Merge_df['Ottoneu ID'] = Full_Merge_df['Ottoneu ID'].str.replace('.0','')

In [ ]:
Full_Merge_sorted_df = Full_Merge_df.sort_values(by=['TOTAL_SGP_Val_mySGP'],ascending=False)

In [ ]:
def quick_plotting_fn(quick_plot):
    fig = plt.figure(figsize=(10,5))
    ax1 = fig.add_subplot(111)
    ax1.errorbar(
        y=quick_plot['Name'],
        x=quick_plot['Ottoneu_Avg'],
        xerr=[
            quick_plot['Ottoneu_Avg']-quick_plot['Ottoneu_Min'],
            quick_plot['Ottoneu_Max']-quick_plot['Ottoneu_Avg']
        ],
        fmt='o',
        color='blue',
        label='4x4 Avg, Min, Max'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Ottoneu_Med'],
        marker='^',
        color='black',
        label='4x4 Median'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Salary'],
        marker='$\\$$',
        s=150,
        color='red',
        label='Current Salary'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Dollars_FGAV'],
        marker='$FG$',
        s=150,
        color='green',
        label='FG Auction Calc.'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Dollars_Vibbot'],
        marker='+',
        s=100,
        color='tab:brown',
        label='Secret Sauce V'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['TOTAL_SGP_Val_mySGP'],
        marker='*',
        s=100,
        color='tab:pink',
        label='Secret Sauce C'
    )
    ax1.set_ylabel('Player')
    ax1.invert_yaxis()
    ax1.set_xlabel('Dollars')
    ax1.legend(loc='lower right')
    plt.tick_params(axis='y', which='major', labelsize=7)
    plt.show()

In [ ]:
This_team_df = Full_Merge_sorted_df[Full_Merge_sorted_df['TeamID'] == TeamID]

In [ ]:
quick_plotting_fn(This_team_df)

In [ ]:
Team_hitter_groupby = Full_Merge_sorted_df[~Full_Merge_sorted_df['Repl_Pos_mySGP'].isin(['SP', 'RP'])].groupby(['Team Name'])[[
    # 'Name',
    # 'Team',
    # 'Ottoneu Positions',
    # 'Repl_Pos_mySGP',
    'G_mySGP',
    'HR_mySGP',
    'R_mySGP',
    'OBP_mySGP',
    'SLG_mySGP',
    # 'IP_mySGP',
    'R_SGP_raw_mySGP',
    'HR_SGP_raw_mySGP',
    'OBP_SGP_raw_mySGP',
    'SLG_SGP_raw_mySGP',
    'R_SGP_norm_mySGP',
    'HR_SGP_norm_mySGP',
    'OBP_SGP_norm_mySGP',
    'SLG_SGP_norm_mySGP',
    # 'SO_mySGP',
    # 'HR9_mySGP',
    # 'ERA_mySGP',
    # 'WHIP_mySGP',
    # 'SO_SGP_Val_mySGP',
    # 'HR9_SGP_Val_mySGP',
    # 'ERA_SGP_Val_mySGP',
    # 'WHIP_SGP_Val_mySGP',
    'TOTAL_SGP_raw_mySGP',
    'TOTAL_SGP_mySGP',
    # 'TOTAL_SGP_Rank_mySGP',
    # 'TOTAL_SGP_Val_mySGP',
    # 'Dollars_FGAV',
    # 'Dollars_Vibbot',
    # 'Ottoneu_Avg',
    # 'Ottoneu_Med',
    # 'Ottoneu_Min',
    # 'Ottoneu_Max',
    # 'Ottoneu_L10',
    # 'Roster%',
    # 'TeamID',
    # 'Team Name',
    # 'Salary'
    ]].sum()



In [ ]:
Team_hitter_groupby

In [ ]:
pd.DataFrame(((Team_hitter_groupby.loc['Largely Indistinguishables']/Team_hitter_groupby.sum())*100)[['G_mySGP',
    'HR_mySGP',
    'R_mySGP',
    'OBP_mySGP',
    'SLG_mySGP']]).T

In [ ]:
fig = plt.figure(figsize=(10,5))
ax1 = fig.add_subplot(111)
sns.swarmplot(pd.DataFrame(((Team_hitter_groupby.loc['Largely Indistinguishables']/Team_hitter_groupby.sum())*100)[['G_mySGP',
    'HR_mySGP',
    'R_mySGP',
    'OBP_mySGP',
    'SLG_mySGP',
    ]]).T,orient='h',ax=ax1,color='tab:pink',size=10)
sns.swarmplot(((Team_hitter_groupby/Team_hitter_groupby.sum())*100)[['G_mySGP',
    'HR_mySGP',
    'R_mySGP',
    'OBP_mySGP',
    'SLG_mySGP']],orient='h',ax=ax1)

In [ ]:
# fig = plt.figure(figsize=(10,5))
# ax1 = fig.add_subplot(111)
# sns.swarmplot(pd.DataFrame(((Team_hitter_groupby.loc['Largely Indistinguishables']/Team_hitter_groupby.sum())*100)[['R_SGP_raw_mySGP',
#     'HR_SGP_raw_mySGP',
#     'OBP_SGP_raw_mySGP',
#     'SLG_SGP_raw_mySGP',
#     'TOTAL_SGP_raw_mySGP']]).T,orient='h',ax=ax1,color='tab:pink',size=10)
# sns.swarmplot(((Team_hitter_groupby/Team_hitter_groupby.sum())*100)[['R_SGP_raw_mySGP',
#     'HR_SGP_raw_mySGP',
#     'OBP_SGP_raw_mySGP',
#     'SLG_SGP_raw_mySGP',
#     'TOTAL_SGP_raw_mySGP']],orient='h',ax=ax1)

In [ ]:
fig = plt.figure(figsize=(10,5))
ax1 = fig.add_subplot(111)
sns.swarmplot(pd.DataFrame(((Team_hitter_groupby.loc['Largely Indistinguishables']/Team_hitter_groupby.sum())*100)[['R_SGP_norm_mySGP',
    'HR_SGP_norm_mySGP',
    'OBP_SGP_norm_mySGP',
    'SLG_SGP_norm_mySGP',
    'TOTAL_SGP_mySGP']]).T,orient='h',ax=ax1,color='tab:pink',size=10)
sns.swarmplot(((Team_hitter_groupby/Team_hitter_groupby.sum())*100)[['R_SGP_norm_mySGP',
    'HR_SGP_norm_mySGP',
    'OBP_SGP_norm_mySGP',
    'SLG_SGP_norm_mySGP',
    'TOTAL_SGP_mySGP']],orient='h',ax=ax1)

In [ ]:
Team_pitcher_groupby = Full_Merge_sorted_df[Full_Merge_sorted_df['Repl_Pos_mySGP'].isin(['SP', 'RP'])].groupby(['Team Name'])[[
    # 'Name',
    # 'Team',
    # 'Ottoneu Positions',
    # 'Repl_Pos_mySGP',
    # 'G_mySGP',
    # 'HR_mySGP',
    # 'R_mySGP',
    # 'OBP_mySGP',
    # 'SLG_mySGP',
    'IP_mySGP',
    # 'R_SGP_raw_mySGP',
    # 'HR_SGP_raw_mySGP',
    # 'OBP_SGP_raw_mySGP',
    # 'SLG_SGP_raw_mySGP',
    # 'R_SGP_norm_mySGP',
    # 'HR_SGP_norm_mySGP',
    # 'OBP_SGP_norm_mySGP',
    # 'SLG_SGP_norm_mySGP',
    'SO_mySGP',
    'HR9_mySGP',
    'ERA_mySGP',
    'WHIP_mySGP',
    'SO_SGP_raw_mySGP',
    'HR9_SGP_raw_mySGP',
    'ERA_SGP_raw_mySGP',
    'WHIP_SGP_raw_mySGP',
    'SO_SGP_norm_mySGP',
    'HR9_SGP_norm_mySGP',
    'ERA_SGP_norm_mySGP',
    'WHIP_SGP_norm_mySGP',
    'TOTAL_SGP_raw_mySGP',
    'TOTAL_SGP_mySGP',
    # 'TOTAL_SGP_Rank_mySGP',
    # 'TOTAL_SGP_Val_mySGP',
    # 'Dollars_FGAV',
    # 'Dollars_Vibbot',
    # 'Ottoneu_Avg',
    # 'Ottoneu_Med',
    # 'Ottoneu_Min',
    # 'Ottoneu_Max',
    # 'Ottoneu_L10',
    # 'Roster%',
    # 'TeamID',
    # 'Team Name',
    # 'Salary'
    ]].sum()

In [ ]:
fig = plt.figure(figsize=(10,5))
ax1 = fig.add_subplot(111)
sns.swarmplot(pd.DataFrame(((Team_pitcher_groupby.loc['Largely Indistinguishables']/Team_pitcher_groupby.sum())*100)[['IP_mySGP',
    'SO_mySGP',
    'HR9_mySGP',
    'ERA_mySGP',
    'WHIP_mySGP',
    ]]).T,orient='h',ax=ax1,color='tab:pink',size=10)
sns.swarmplot(((Team_pitcher_groupby/Team_pitcher_groupby.sum())*100)[['IP_mySGP',
    'SO_mySGP',
    'HR9_mySGP',
    'ERA_mySGP',
    'WHIP_mySGP']],orient='h',ax=ax1)

In [ ]:
# fig = plt.figure(figsize=(10,5))
# ax1 = fig.add_subplot(111)
# sns.swarmplot(pd.DataFrame(((Team_pitcher_groupby.loc['Largely Indistinguishables']/Team_pitcher_groupby.sum())*100)[['SO_SGP_raw_mySGP',
#     'HR9_SGP_raw_mySGP',
#     'ERA_SGP_raw_mySGP',
#     'WHIP_SGP_raw_mySGP',
#     'TOTAL_SGP_raw_mySGP']]).T,orient='h',ax=ax1,color='tab:pink',size=10)
# sns.swarmplot(((Team_pitcher_groupby/Team_pitcher_groupby.sum())*100)[['SO_SGP_raw_mySGP',
#     'HR9_SGP_raw_mySGP',
#     'ERA_SGP_raw_mySGP',
#     'WHIP_SGP_raw_mySGP',
#     'TOTAL_SGP_raw_mySGP']],orient='h',ax=ax1)

In [ ]:
fig = plt.figure(figsize=(10,5))
ax1 = fig.add_subplot(111)
sns.swarmplot(pd.DataFrame(((Team_pitcher_groupby.loc['Largely Indistinguishables']/Team_pitcher_groupby.sum())*100)[['SO_SGP_norm_mySGP',
    'HR9_SGP_norm_mySGP',
    'ERA_SGP_norm_mySGP',
    'WHIP_SGP_norm_mySGP',
    'TOTAL_SGP_mySGP']]).T,orient='h',ax=ax1,color='tab:pink',size=10)
sns.swarmplot(((Team_pitcher_groupby/Team_pitcher_groupby.sum())*100)[['SO_SGP_norm_mySGP',
    'HR9_SGP_norm_mySGP',
    'ERA_SGP_norm_mySGP',
    'WHIP_SGP_norm_mySGP',
    'TOTAL_SGP_mySGP']],orient='h',ax=ax1)

# Fully Replacement Roster

In [ ]:
Target_Stats_df.head()

In [ ]:
Replacement_Level_Recalc_df[['Ottoneu Positions','R_mySGP','HR_mySGP','OBP_mySGP','SLG_mySGP','SO_mySGP','HR9_mySGP','ERA_mySGP','WHIP_mySGP']]


In [ ]:
Replacement_Level_Orig_df[['Ottoneu Positions','G','R','HR','OBP','SLG','SO','HR9','ERA','WHIP','IP']]

In [ ]:
Replacement_Level_Recalc_df[['Ottoneu Positions','R_SGP_raw_mySGP','HR_SGP_raw_mySGP','OBP_SGP_raw_mySGP','SLG_SGP_raw_mySGP','SO_SGP_raw_mySGP','HR9_SGP_raw_mySGP','ERA_SGP_raw_mySGP','WHIP_SGP_raw_mySGP']]


In [ ]:
Replacement_Level_Orig_df[['Ottoneu Positions','R_SGP_raw','HR_SGP_raw','OBP_SGP_raw','SLG_SGP_raw','SO_SGP_raw','HR9_SGP_raw','ERA_SGP_raw','WHIP_SGP_raw','TOTAL_SGP_raw']]


In [ ]:
Replacement_Level_Recalc_df[['Ottoneu Positions','R_SGP_norm_mySGP','HR_SGP_norm_mySGP','OBP_SGP_norm_mySGP','SLG_SGP_norm_mySGP','SO_SGP_norm_mySGP','HR9_SGP_norm_mySGP','ERA_SGP_norm_mySGP','WHIP_SGP_norm_mySGP']]

In [ ]:
Replacement_Level_Recalc_df[['Ottoneu Positions','mR_FGAV','mHR_FGAV','mOBP_FGAV','mSLG_FGAV','mSO_FGAV','mERA_FGAV','mWHIP_FGAV']]

In [ ]:
Both_Replacement_Level_df = Replacement_Level_Recalc_df.merge(Replacement_Level_Orig_df,how='outer').set_index('Ottoneu Positions')

In [ ]:
Both_Replacement_Level_df.columns

In [ ]:
Both_Replacement_Level_df[['R','R_mySGP','TOTAL_SGP_raw']]

In [ ]:
sns.lmplot(Both_Replacement_Level_df[['HR','HR_mySGP']],x='HR',y='HR_mySGP')

In [ ]:
Both_Replacement_Level_df.loc['MI'].to_dict()

In [ ]:
Games_IP_Targets

In [ ]:
Batting_value_columns = {
    'G':['TOTAL_SGP_Val_mySGP','TOTAL_SGP_raw'],
    'R':['R_mySGP','R_SGP_raw_mySGP','R_SGP_norm_mySGP','R_SGP_Val_mySGP','mR_FGAV'],
    'HR':['HR_mySGP','HR_SGP_raw_mySGP','HR_SGP_norm_mySGP','HR_SGP_Val_mySGP','mHR_FGAV'],
    'OBP':['OBP_mySGP','OBP_SGP_raw_mySGP','OBP_SGP_norm_mySGP','OBP_SGP_Val_mySGP','mOBP_FGAV'],
    'SLG':['SLG_mySGP','SLG_SGP_raw_mySGP','SLG_SGP_norm_mySGP','SLG_SGP_Val_mySGP','mSLG_FGAV']
}

Pitching_value_columns = {
    'IP':['TOTAL_SGP_Val_mySGP','TOTAL_SGP_raw'],
    'SO':['SO_mySGP','SO_SGP_raw_mySGP','SO_SGP_norm_mySGP','SO_SGP_Val_mySGP','mSO_FGAV'],
    'HR9':['HR9_mySGP','HR9_SGP_raw_mySGP','HR9_SGP_norm_mySGP','HR9_SGP_Val_mySGP','mHR_FGAV'],
    'ERA':['ERA_mySGP','ERA_SGP_raw_mySGP','ERA_SGP_norm_mySGP','ERA_SGP_Val_mySGP','mERA_FGAV'],
    'WHIP':['WHIP_mySGP','WHIP_SGP_raw_mySGP','WHIP_SGP_norm_mySGP','WHIP_SGP_Val_mySGP','mWHIP_FGAV']
}
count_cols_hitting = ['G'] + Batting_value_columns['G'] + \
             ['R'] + Batting_value_columns['R'] + \
             ['HR'] + Batting_value_columns['HR'] + \
             ['OBP_SGP_raw_mySGP','OBP_SGP_norm_mySGP','OBP_SGP_Val_mySGP','mOBP_FGAV'] + \
             ['SLG_SGP_raw_mySGP','SLG_SGP_norm_mySGP','SLG_SGP_Val_mySGP','mSLG_FGAV']
rate_cols_hitting = ['OBP','OBP_mySGP','SLG','SLG_mySGP']

count_cols_pitching = ['IP'] + Pitching_value_columns['IP'] + \
             ['SO'] + Pitching_value_columns['SO'] + \
             ['HR9_SGP_raw_mySGP','HR9_SGP_norm_mySGP','HR9_SGP_Val_mySGP','mHR_FGAV'] + \
             ['ERA_SGP_raw_mySGP','ERA_SGP_norm_mySGP','ERA_SGP_Val_mySGP','mERA_FGAV'] + \
             ['WHIP_SGP_raw_mySGP','WHIP_SGP_norm_mySGP','WHIP_SGP_Val_mySGP','mWHIP_FGAV']
rate_cols_pitching = ['HR9','HR9_mySGP','ERA','ERA_mySGP','WHIP','WHIP_mySGP']

Replacement_level_team_df = pd.DataFrame()
for pos,starters in Start_Spots.items():
    This_Pos_Replacement_Level = Both_Replacement_Level_df.loc[pos]
    if pos == 'MI':
        This_Pos_Replacement_Level = Both_Replacement_Level_df.loc['SS']
    This_Pos_Stat_Cats = Pitching_value_columns if pos in Pitching_Pos else Batting_value_columns
    count_cols = count_cols_pitching if pos in Pitching_Pos else count_cols_hitting
    rate_cols = rate_cols_pitching if pos in Pitching_Pos else rate_cols_hitting
    pos_G_IP_target = Games_IP_Targets['IP'] if pos in Pitching_Pos else Games_IP_Targets[pos]
    # print(pos_G_IP_target)
    # print((This_Pos_Replacement_Level['IP'] if pos in Pitching_Pos else This_Pos_Replacement_Level['G']))
    pos_count_norm = (pos_G_IP_target/(This_Pos_Replacement_Level['IP'] if pos in Pitching_Pos else This_Pos_Replacement_Level['G']))
    # print(pos, pos_G_IP_target)
    # print((This_Pos_Replacement_Level['G']))
    This_Pos_Replacement_Stats = {'Ottoneu Positions':pos,'Hitter Pitcher':('Pitcher' if pos in Pitching_Pos else 'Hitter')}
    for stat_cat, stat_cols in This_Pos_Stat_Cats.items():
        #print(stat_cat, stat_cols)
        this_stat_count_cols = list(set([stat_cat]+stat_cols) & set(count_cols))
        this_stat_rate_cols = list(set([stat_cat]+stat_cols) & set(rate_cols))
        This_Pos_Replacement_Starters = (This_Pos_Replacement_Level[this_stat_count_cols]*pos_count_norm).to_dict() | This_Pos_Replacement_Level[this_stat_rate_cols].apply(lambda value: [value]*starters).to_dict()
        # if pos == 'MI':
        #     display((This_Pos_Replacement_Level[this_stat_count_cols]*pos_count_norm).to_dict())
        #     display(This_Pos_Replacement_Level[this_stat_rate_cols].apply(lambda value: [value]*starters).to_dict())
        #     display(This_Pos_Replacement_Starters)
        #print(This_Pos_Replacement_Level[this_stat_rate_cols].apply(lambda value: [value]*starters).to_dict())
        This_Pos_Replacement_Stats = This_Pos_Replacement_Stats | This_Pos_Replacement_Starters
    Replacement_level_team_df = pd.concat([
        Replacement_level_team_df,
        pd.DataFrame([This_Pos_Replacement_Stats])
    ])
display(Replacement_level_team_df)
    

In [ ]:
Replacement_level_team_df.set_index(['Ottoneu Positions','Hitter Pitcher']).columns

In [ ]:
Target_Stats_df.columns

In [ ]:
Target_Stats_df['TOTAL_SGP_raw'] = Target_Stats_df['Target Pts']*len(Scoring_Categories)
Target_Stats_df['R_mySGP_Team'] = Target_Stats_df['R']
Target_Stats_df['R_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['R_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['R_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mR_FGAV_Team'] = 1.
Target_Stats_df['HR_mySGP_Team'] = Target_Stats_df['HR']
Target_Stats_df['HR_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['HR_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['HR_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mHR_FGAV_Team'] = 1.
Target_Stats_df['OBP_Team'] = Target_Stats_df['OBP']
Target_Stats_df['OBP_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['OBP_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['OBP_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mOBP_FGAV_Team'] = 1.
Target_Stats_df['SLG_Team'] = Target_Stats_df['SLG']
Target_Stats_df['SLG_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['SLG_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['SLG_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mSLG_FGAV_Team'] = 1.

Target_Stats_df['SO_mySGP_Team'] = Target_Stats_df['SO']
Target_Stats_df['SO_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['SO_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['SO_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mSO_FGAV_Team'] = 1.
Target_Stats_df['HR9_Team'] = Target_Stats_df['HR9']
Target_Stats_df['HR9_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['HR9_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['HR9_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mHR_FGAV_Team'] = 1.
Target_Stats_df['ERA_Team'] = Target_Stats_df['ERA']
Target_Stats_df['ERA_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['ERA_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['ERA_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mERA_FGAV_Team'] = 1.
Target_Stats_df['WHIP_Team'] = Target_Stats_df['WHIP']
Target_Stats_df['WHIP_SGP_Val_mySGP_Team'] = Target_Stats_df['CAT_SGP_Val']
Target_Stats_df['WHIP_SGP_raw_mySGP_Team'] = Target_Stats_df['Target Pts']
Target_Stats_df['WHIP_SGP_norm_mySGP_Team'] = 1.
Target_Stats_df['mWHIP_FGAV_Team'] = 1.

In [ ]:
# bugged_cols = rate_cols_hitting+rate_cols_pitching+[
#     'mR_FGAV', 'mHR_FGAV', 'mOBP_FGAV', 'mSLG_FGAV', 'mSO_FGAV', 'mHR_FGAV', 'mERA_FGAV', 'mWHIP_FGAV', 
#     'R_SGP_norm_mySGP','HR_SGP_norm_mySGP', 'OBP_SGP_norm_mySGP', 'SLG_SGP_norm_mySGP',
#     'SO_SGP_norm_mySGP','HR9_SGP_norm_mySGP', 'ERA_SGP_norm_mySGP', 'WHIP_SGP_norm_mySGP', 
#     'R_SGP_raw_mySGP','HR_SGP_raw_mySGP', 'OBP_SGP_raw_mySGP', 'SLG_SGP_raw_mySGP',
#     'SO_SGP_raw_mySGP','HR9_SGP_raw_mySGP', 'ERA_SGP_raw_mySGP', 'WHIP_SGP_raw_mySGP'
# ]

In [ ]:
# for Stat in list(Replacement_level_team_df[Replacement_level_team_df['Hitter Pitcher'] == 'Hitter'].dropna(axis=1).set_index(['Ottoneu Positions','Hitter Pitcher']).columns):
#     if Stat in bugged_cols:
#         continue
#     fig = plt.figure(figsize=(10,5))
#     ax1 = fig.add_subplot(111)
#     sns.scatterplot(Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == 'Hitter'],x='Hitter Pitcher',y=Stat,color='Red',label='Target',marker='D',ax=ax1)
#     sns.histplot(Replacement_level_team_df[Replacement_level_team_df['Hitter Pitcher'] == 'Hitter'],x='Hitter Pitcher',weights=Stat,hue='Ottoneu Positions', multiple='stack',ax=ax1)
#     plt.tight_layout()
#     plt.show()

In [ ]:
This_team_df[['Name','Team','Ottoneu Positions','Repl_Pos_mySGP','TOTAL_SGP_mySGP']]

In [ ]:
Both_Replacement_Level_df[['R_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP']]

In [ ]:
This_team_df[['Name','Ottoneu Positions','Repl_Pos_mySGP','R_SGP_repl_mySGP','TOTAL_SGP_mySGP','TOTAL_SGP_Val_mySGP','TeamID','Team Name']]

In [ ]:
Both_Replacement_Level_df.loc['C'].keys()

In [ ]:
This_team_df['TeamID'].unique()[0]

In [ ]:
list(Both_Replacement_Level_df.columns)

In [ ]:
All_teams_replacement_level = pd.DataFrame()
for teamID in Full_Merge_sorted_df['TeamID'].dropna().unique():
    One_team_plus_Replacement_level_df = pd.DataFrame()
    This_team_df = Full_Merge_sorted_df[Full_Merge_sorted_df['TeamID'] == teamID]
    print(This_team_df['Team Name'].unique()[0])
    This_team_df['Repl_Pos_mySGP'] = This_team_df['Repl_Pos_mySGP'].fillna('MiLB')
    This_team_df['Repl_Pos_mySGP'] = This_team_df['Repl_Pos_mySGP'].replace({'Util/SP':'Util'})
    #display(This_team_df[['Name','Team','Ottoneu Positions','Repl_Pos_mySGP','TOTAL_SGP_mySGP','TOTAL_SGP_Val_mySGP']])
    team_pos_override = Total_Roster_Spots.copy()
    for pos,spots in team_pos_override.items(): #MI after 2B and SS
        This_Pos_Replacement_Level = Both_Replacement_Level_df.loc[pos]
        This_Pos_one_team = This_team_df[This_team_df['Repl_Pos_mySGP'].apply(lambda x : pos in x.split('/'))]
        #  if pos == 'MI':
        #     This_Pos_Replacement_Level = Both_Replacement_Level_df.loc['SS']
        #     This_team_2B = This_team_df[This_team_df['Repl_Pos_mySGP'].apply(lambda x : '2B' in x.split('/'))]
        #     This_team_SS = This_team_df[This_team_df['Repl_Pos_mySGP'].apply(lambda x : 'SS' in x.split('/'))]
        #     This_Pos_one_team = pd.concat([This_team_2B,This_team_SS]).sort_values('TOTAL_SGP_mySGP',ascending=False)
        if pos == 'Util':
            This_Pos_one_team = This_team_df[~This_team_df['Repl_Pos_mySGP'].isin(['SP','RP','SP/RP'])]
        
        if One_team_plus_Replacement_level_df.shape[0] != 0:
            This_Pos_one_team = This_Pos_one_team[(~This_Pos_one_team['Ottoneu ID'].isin(list(One_team_plus_Replacement_level_df['Ottoneu ID'].unique())))]
        This_Pos_one_team['Roster_Spot'] = pos
        One_team_plus_Replacement_level_df = pd.concat([
            One_team_plus_Replacement_level_df,
            This_Pos_one_team.sort_values('TOTAL_SGP_mySGP',ascending=False)#.head(Total_Roster_Spots.get(pos))
        ])
        Num_Repl_this_pos = spots - One_team_plus_Replacement_level_df[One_team_plus_Replacement_level_df['Roster_Spot'] == pos].shape[0]
        if Num_Repl_this_pos < 0:
            print('*** ', pos,'Over allocated ***')

        # print(pos)
        # print('Roster Spots:', spots)
        # print('Num Replacement:', Num_Repl_this_pos)
        # print('Num Starters:', Start_Spots[pos])
        while Num_Repl_this_pos > 0:
            Num_Repl_this_pos-=1
            Replacement_Player = {
                'Name':f'Replacement {pos} {Num_Repl_this_pos+1}',
                'Ottoneu Positions':pos,
                'Repl_Pos_mySGP':pos,
                'Roster_Spot':pos,
                'G_mySGP':This_Pos_Replacement_Level['G'],
                'PA_mySGP':This_Pos_Replacement_Level['PA'],
                'AB_mySGP':This_Pos_Replacement_Level['AB'],
                'H_mySGP':This_Pos_Replacement_Level['H'],
                '1B_mySGP':This_Pos_Replacement_Level['1B'],
                '2B_mySGP':This_Pos_Replacement_Level['2B'],
                '3B_mySGP':This_Pos_Replacement_Level['3B'],
                'HR_mySGP':This_Pos_Replacement_Level['HR_mySGP'],
                'R_mySGP':This_Pos_Replacement_Level['R_mySGP'],
                'RBI_mySGP':This_Pos_Replacement_Level['RBI'],
                'BB_mySGP':This_Pos_Replacement_Level['BB'],
                'HBP_mySGP':This_Pos_Replacement_Level['HBP'],
                'SF_mySGP':This_Pos_Replacement_Level['SF'],
                'WAR_mySGP':This_Pos_Replacement_Level['WAR'],
                'ADP_mySGP':This_Pos_Replacement_Level['ADP'],
                'POS_mySGP':pos,
                #'Ottoneu ID':,
                'OBP_mySGP':This_Pos_Replacement_Level['OBP_mySGP'],
                #'TB_mySGP':This_Pos_Replacement_Level['TB'],
                'SLG_mySGP':This_Pos_Replacement_Level['SLG_mySGP'],
                #'Pos Place_mySGP':This_Pos_Replacement_Level['Pos Place_mySGP'],
                'R_SGP_raw_mySGP':This_Pos_Replacement_Level['R_SGP_raw_mySGP'],
                'HR_SGP_raw_mySGP':This_Pos_Replacement_Level['HR_SGP_raw_mySGP'],
                'OBP_SGP_raw_mySGP':This_Pos_Replacement_Level['OBP_SGP_raw_mySGP'],
                'SLG_SGP_raw_mySGP':This_Pos_Replacement_Level['SLG_SGP_raw_mySGP'],
                'TOTAL_SGP_raw_mySGP':This_Pos_Replacement_Level['TOTAL_SGP_raw'],
                'R_SGP_repl_mySGP':This_Pos_Replacement_Level['R_SGP_raw_mySGP'],
                'Repl_Pos_mySGP':pos,
                'R_SGP_norm_mySGP':This_Pos_Replacement_Level['R_SGP_norm_mySGP'],
                'HR_SGP_repl_mySGP':This_Pos_Replacement_Level['HR_SGP_raw_mySGP'],
                'HR_SGP_norm_mySGP':This_Pos_Replacement_Level['HR_SGP_norm_mySGP'],
                'OBP_SGP_repl_mySGP':This_Pos_Replacement_Level['OBP_SGP_raw_mySGP'],
                'OBP_SGP_norm_mySGP':This_Pos_Replacement_Level['OBP_SGP_norm_mySGP'],
                'SLG_SGP_repl_mySGP':This_Pos_Replacement_Level['SLG_SGP_raw_mySGP'],
                'SLG_SGP_norm_mySGP':This_Pos_Replacement_Level['SLG_SGP_norm_mySGP'],
                #'TOTAL_SGP_mySGP':This_Pos_Replacement_Level['TOTAL_SGP_raw'],
                #'TOTAL_SGP_Rank_mySGP':This_Pos_Replacement_Level['TOTAL_SGP_Rank_mySGP'],
                'TOTAL_SGP_Val_mySGP':This_Pos_Replacement_Level['TOTAL_SGP_Val_mySGP'],
                'R_SGP_Val_mySGP':This_Pos_Replacement_Level['R_SGP_Val_mySGP'],
                'HR_SGP_Val_mySGP':This_Pos_Replacement_Level['HR_SGP_Val_mySGP'],
                'OBP_SGP_Val_mySGP':This_Pos_Replacement_Level['OBP_SGP_Val_mySGP'],
                'SLG_SGP_Val_mySGP':This_Pos_Replacement_Level['SLG_SGP_Val_mySGP'],
                'W_mySGP':This_Pos_Replacement_Level['W'],
                'L_mySGP':This_Pos_Replacement_Level['L'],
                'QS_mySGP':This_Pos_Replacement_Level['QS'],
                'GS_mySGP':This_Pos_Replacement_Level['GS'],
                'SV_mySGP':This_Pos_Replacement_Level['SV'],
                'HLD_mySGP':This_Pos_Replacement_Level['HLD'],
                'IP_mySGP':This_Pos_Replacement_Level['IP'],
                'TBF_mySGP':This_Pos_Replacement_Level['TBF'],
                'ER_mySGP':This_Pos_Replacement_Level['ER'],
                'SO_mySGP':This_Pos_Replacement_Level['SO_mySGP'],
                'HR9_mySGP':This_Pos_Replacement_Level['HR9_mySGP'],
                'ERA_mySGP':This_Pos_Replacement_Level['ERA_mySGP'],
                'WHIP_mySGP':This_Pos_Replacement_Level['WHIP_mySGP'],
                'SO_SGP_raw_mySGP':This_Pos_Replacement_Level['SO_SGP_raw_mySGP'],
                'HR9_SGP_raw_mySGP':This_Pos_Replacement_Level['HR9_SGP_raw_mySGP'],
                'ERA_SGP_raw_mySGP':This_Pos_Replacement_Level['ERA_SGP_raw_mySGP'],
                'WHIP_SGP_raw_mySGP':This_Pos_Replacement_Level['WHIP_SGP_raw_mySGP'],
                'SO_SGP_repl_mySGP':This_Pos_Replacement_Level['SO_SGP_raw_mySGP'],
                'SO_SGP_norm_mySGP':This_Pos_Replacement_Level['SO_SGP_norm_mySGP'],
                'HR9_SGP_repl_mySGP':This_Pos_Replacement_Level['HR9_SGP_raw_mySGP'],
                'HR9_SGP_norm_mySGP':This_Pos_Replacement_Level['HR9_SGP_norm_mySGP'],
                'ERA_SGP_repl_mySGP':This_Pos_Replacement_Level['ERA_SGP_raw_mySGP'],
                'ERA_SGP_norm_mySGP':This_Pos_Replacement_Level['ERA_SGP_norm_mySGP'],
                'WHIP_SGP_repl_mySGP':This_Pos_Replacement_Level['WHIP_SGP_raw_mySGP'],
                'WHIP_SGP_norm_mySGP':This_Pos_Replacement_Level['WHIP_SGP_norm_mySGP'],
                'SO_SGP_Val_mySGP':This_Pos_Replacement_Level['SO_SGP_Val_mySGP'],
                'HR9_SGP_Val_mySGP':This_Pos_Replacement_Level['HR9_SGP_Val_mySGP'],
                'ERA_SGP_Val_mySGP':This_Pos_Replacement_Level['ERA_SGP_Val_mySGP'],
                'WHIP_SGP_Val_mySGP':This_Pos_Replacement_Level['WHIP_SGP_Val_mySGP'],
                'POS_FGAV':pos,
                'ADP_FGAV':This_Pos_Replacement_Level['ADP'],
                'PA_FGAV':This_Pos_Replacement_Level['PA'],
                'mR_FGAV':This_Pos_Replacement_Level['mR_FGAV'],
                'mHR_FGAV':This_Pos_Replacement_Level['mHR_FGAV'],
                'mOBP_FGAV':This_Pos_Replacement_Level['mOBP_FGAV'],
                'mSLG_FGAV':This_Pos_Replacement_Level['mSLG_FGAV'],
                #'PTS_FGAV':This_Pos_Replacement_Level['PTS_FGAV'],
                #'aPOS_FGAV':This_Pos_Replacement_Level['aPOS_FGAV'],
                #'Dollars_FGAV':This_Pos_Replacement_Level['Dollars_FGAV'],
                #'NameASCII_FGAV':This_Pos_Replacement_Level['NameASCII_FGAV'],
                #'MLBAMID_FGAV':This_Pos_Replacement_Level['MLBAMID_FGAV'],
                'IP_FGAV':This_Pos_Replacement_Level['IP'],
                'mERA_FGAV':This_Pos_Replacement_Level['mERA_FGAV'],
                'mWHIP_FGAV':This_Pos_Replacement_Level['mWHIP_FGAV'],
                'mSO_FGAV':This_Pos_Replacement_Level['mSO_FGAV'],
                #'Dollars_Vibbot':This_Pos_Replacement_Level['Dollars_Vibbot'],
                # 'Ottoneu_Avg':This_Pos_Replacement_Level['Ottoneu_Avg'],
                # 'Ottoneu_Med':This_Pos_Replacement_Level['Ottoneu_Med'],
                # 'Ottoneu_Min':This_Pos_Replacement_Level['Ottoneu_Min'],
                # 'Ottoneu_Max':This_Pos_Replacement_Level['Ottoneu_Max'],
                # 'Ottoneu_L10':This_Pos_Replacement_Level['Ottoneu_L10'],
                # 'Roster%':This_Pos_Replacement_Level['Roster%'],
                'TeamID':This_team_df['TeamID'].unique()[0],
                'Team Name':This_team_df['Team Name'].unique()[0],
                'Salary':1.
            }
            One_team_plus_Replacement_level_df = pd.concat([
                One_team_plus_Replacement_level_df,
                pd.DataFrame([Replacement_Player])
            ])

    One_team_plus_Replacement_level_df = pd.concat([
        One_team_plus_Replacement_level_df,
        This_team_df[This_team_df['Repl_Pos_mySGP'] == 'MiLB']
    ])
    One_team_plus_Replacement_level_df.reset_index(drop=True,inplace=True)
    #display(One_team_plus_Replacement_level_df[['Name','Team','Ottoneu Positions','Repl_Pos_mySGP','Roster_Spot','TOTAL_SGP_raw_mySGP','TOTAL_SGP_mySGP','TOTAL_SGP_Val_mySGP']])
    All_teams_replacement_level = pd.concat(
        [
            All_teams_replacement_level,
            One_team_plus_Replacement_level_df
        ]
    )

# Manual Review of each team

In [ ]:
Beavers_df = All_teams_replacement_level[All_teams_replacement_level['Team Name'] == '🦫 Beavers in Scoring Position!']

Beavers_df = Beavers_df[~Beavers_df['Name'].isin([
    'Replacement 2B 3',
    'Replacement 2B 2',
    #'Replacement 2B 1',
    'Replacement RP 6',
    'Replacement RP 5',
    'Replacement RP 4',
    'Replacement RP 3',
    'Replacement RP 2',

])].reset_index(drop=True)

# display(Beavers_df[['Name','Team','Ottoneu Positions','Repl_Pos_mySGP','Roster_Spot','TOTAL_SGP_raw_mySGP','TOTAL_SGP_mySGP','TOTAL_SGP_Val_mySGP']])

In [ ]:
All_teams_replacement_level = pd.concat(
    [
        All_teams_replacement_level[(All_teams_replacement_level['Team Name'] != '🦫 Beavers in Scoring Position!')],
        Beavers_df
    ]).reset_index(drop=True)

#Beavers_df = All_teams_replacement_level[All_teams_replacement_level['Team Name'] == '🦫 Beavers in Scoring Position!']

In [ ]:
Second_Best_Lung_df = All_teams_replacement_level[(All_teams_replacement_level['Team Name'] == "Jack Klugman's Second Best Lung")]

In [ ]:
Second_Best_Lung_df = Second_Best_Lung_df[
    ~Second_Best_Lung_df['Name'].isin([
'Replacement 2B 3',
'Replacement RP 3',
#'Replacement OF 3'
    ])].reset_index(drop=True)

In [ ]:
All_teams_replacement_level = pd.concat(
    [
        All_teams_replacement_level[(All_teams_replacement_level['Team Name'] != "Jack Klugman's Second Best Lung")],
        Second_Best_Lung_df
    ]).reset_index(drop=True)

In [ ]:
#Second_Best_Lung_df = All_teams_replacement_level[(All_teams_replacement_level['Team Name'] == "Jack Klugman's Second Best Lung")].reset_index()

In [ ]:
Chili_Dog_MVP_df = All_teams_replacement_level[All_teams_replacement_level['Team Name'] == 'Chili Dog MVP'].reset_index(drop=True)

In [ ]:
Chili_Dog_MVP_df = Chili_Dog_MVP_df[
    ~Chili_Dog_MVP_df['Name'].isin([
'Replacement 2B 3',
'Replacement 2B 2',
'Replacement RP 5',
'Replacement RP 4',
#'Replacement OF 3'
    ])].reset_index(drop=True)

In [ ]:
All_teams_replacement_level = pd.concat(
    [
        All_teams_replacement_level[(All_teams_replacement_level['Team Name'] != "Chili Dog MVP")],
        Chili_Dog_MVP_df
    ]).reset_index(drop=True)

In [ ]:
# Chili_Dog_MVP_df = All_teams_replacement_level[All_teams_replacement_level['Team Name'] == 'Chili Dog MVP'].reset_index(drop=True)

In [ ]:
# display(Chili_Dog_MVP_df[['Name','Team','Ottoneu Positions','Repl_Pos_mySGP','Roster_Spot','TOTAL_SGP_raw_mySGP','TOTAL_SGP_mySGP','TOTAL_SGP_Val_mySGP']])

In [ ]:
#list(All_teams_replacement_level.columns)

In [ ]:
All_teams_replacement_level['Roster_Spot_Rank'] = All_teams_replacement_level.groupby(['TeamID','Roster_Spot'])['TOTAL_SGP_raw_mySGP'].rank('first',ascending=False)

In [ ]:
display(All_teams_replacement_level[All_teams_replacement_level['TeamID'] == 55][['Name','Team Name','Ottoneu Positions','Roster_Spot','Roster_Spot_Rank','G_mySGP','IP_mySGP']])

In [ ]:
def group_and_limit_sum(df, limits_dict):
    """
    Groups a DataFrame by columns 'TeamID' and 'Roster_Spot', then in order of rank on 'Roster_Spot_Rank', 
    sums column 'G_mySGP' until a limit (from limits_dict, keyed by 'Roster_Spot') is reached, 
    adding a column 'scale_factor' for the final rank to match the remainder.

    Args:
        df (pd.DataFrame): The input DataFrame with columns 'TeamID', 'Roster_Spot', 'Roster_Spot_Rank', 'G_mySGP'.
        limits_dict (dict): A dictionary where keys match unique values in column 'Roster_Spot' 
                            and values are the sum limits for column 'G_mySGP'.

    Returns:
        pd.DataFrame: The modified DataFrame with added 'cumulative_sum' and 
                      'scale_factor' columns. The 'scale_factor' is 1.0 for 
                      ranks fully included and <= 1.0 for the final, partially 
                      included rank.
    """
    
    # 1. Sort the DataFrame by columns 'TeamID', 'Roster_Spot', and 'Roster_Spot_Rank' (for ranking)
    # This ensures correct ordering within each group before applying cumulative logic.
    df_sorted = df.sort_values(by=['TeamID', 'Roster_Spot', 'Roster_Spot_Rank']).reset_index(drop=True)

    # 2. Define a custom function to apply to each group
    def apply_limit_and_scale(group):
        b_key = group.name[1]
        limit = limits_dict.get(b_key, group['G_mySGP'].sum()) # Default to total sum if key missing
        
        # Calculate cumulative sum
        cum_sum = group['G_mySGP'].cumsum()
        #print(b_key, cum_sum)
        
        # Determine which rows exceed the limit
        exceeded_limit = cum_sum > limit
        
        # Identify the first row that exceeds the limit
        first_exceed_index = exceeded_limit.idxmax() if exceeded_limit.any() else None
        
        # Calculate scale factors
        scale_factors = pd.Series(1.0, index=group.index)
        if first_exceed_index is not None:
            # For rows after the first exceeded, set factor to 0
            scale_factors.loc[first_exceed_index+1:] = 0.0
            
            # For the first exceeded row, calculate the proportional scale factor
            previous_sum = cum_sum.loc[:first_exceed_index-1].iloc[-1] if first_exceed_index != group.index[0] else 0 #.sum()
            remaining = limit - previous_sum
            scale_factors.loc[first_exceed_index] = remaining / group.loc[first_exceed_index, 'G_mySGP']
            
        group['G_mySGP_scale_factor'] = scale_factors
        group['capped_G_mySGP'] = group['G_mySGP'] * group['G_mySGP_scale_factor']
        group['cumulative_capped_G_mySGP_sum'] = group['capped_G_mySGP'].cumsum()
        
        return group

    # 3. Apply the custom function to each group
    # We use groupby on 'TeamID' and 'Roster_Spot' and then apply the custom function
    df_result = df_sorted.groupby(['TeamID', 'Roster_Spot']).apply(apply_limit_and_scale)

    return df_result.sort_index() # Revert to original index order if needed



In [ ]:
Hitters_G_played_df = group_and_limit_sum(All_teams_replacement_level[~All_teams_replacement_level['Roster_Spot'].isin(['SP','RP'])], Games_IP_Targets).reset_index()

In [ ]:
Hitters_G_played_df = Hitters_G_played_df.drop('level_2',axis=1)

In [ ]:
MI_candidates = Hitters_G_played_df[(Hitters_G_played_df['Roster_Spot'].isin(['2B','SS'])) & (Hitters_G_played_df['G_mySGP_scale_factor'] < 1)].reset_index(drop=True)

In [ ]:
MI_candidates = MI_candidates.join(MI_candidates.groupby('TeamID')['TOTAL_SGP_raw_mySGP'].rank('first',ascending=False).reset_index().rename(columns={'TOTAL_SGP_raw_mySGP':'MI_Rank'})).reset_index(drop=True)#[['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'cumulative_capped_G_mySGP_sum']]

In [ ]:
MI_candidates = MI_candidates.drop('index',axis=1)

In [ ]:
MI_candidates['G_MI_avail'] = MI_candidates['G_mySGP'] - MI_candidates['capped_G_mySGP']

In [ ]:
#display(MI_candidates[MI_candidates['TeamID'] == 55][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'G_MI_avail']])#,'MI_scale_factor','capped_MI_G_mySGP']])

In [ ]:
def apply_limit_and_scale_MI(group):
    # Sort by rank
    group = group.sort_values('MI_Rank').reset_index()
    limit = Games_IP_Targets['MI']
    
    # Calculate cumulative sum
    group['cum_sum'] = group['G_MI_avail'].cumsum()
    
    # Identify row where limit is exceeded
    group['over_limit'] = group['cum_sum'] > limit
    
    # Initialize scale factor
    group['MI_scale_factor'] = 1.0
    
    # Find first row over limit
    over_limit_idx = group[group['over_limit']].index

    #print(group)
    
    if not over_limit_idx.empty:
        idx = over_limit_idx[0]
        # Calculate factor for the boundary row
        prev_sum = group.loc[:idx-1, 'G_MI_avail'].sum() if idx > group.index[0] else 0
        remaining = limit - prev_sum
        group.loc[idx, 'MI_scale_factor'] = max(0, remaining / group.loc[idx, 'G_MI_avail'])
        
        # Zero out rows after the limit
        group.loc[idx+1:, 'MI_scale_factor'] = 0.0
        #group.loc[idx+1:, 'G_MI_avail'] = 0.0
        # Cap the breaking row
        #group.loc[idx, 'G_MI_avail'] = max(0, remaining)

    
    group['capped_MI_G_mySGP'] = group['G_MI_avail'] * group['MI_scale_factor']
        
    return group.drop(columns=['cum_sum', 'over_limit'])

# Apply to groups
MI_candidates = MI_candidates.groupby('TeamID').apply(apply_limit_and_scale_MI).reset_index()
#print(MI_candidates)

In [ ]:
#display(MI_candidates[MI_candidates['TeamID'] == 55][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'G_MI_avail','MI_scale_factor','capped_MI_G_mySGP']])

In [ ]:
MI_candidates = MI_candidates.drop(['level_1','index'],axis=1)

In [ ]:
Hitters_G_played_df = Hitters_G_played_df.merge(MI_candidates,how='left')#[['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'G_MI_avail','MI_scale_factor','capped_MI_G_mySGP']]

In [ ]:
#display(Hitters_G_played_df[Hitters_G_played_df['TeamID'] == 55][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'G_MI_avail','MI_scale_factor','capped_MI_G_mySGP']])

In [ ]:
UT_candidates = Hitters_G_played_df[(~Hitters_G_played_df['Roster_Spot'].isin(['SP','RP'])) & (Hitters_G_played_df['G_mySGP_scale_factor'] < 1) & ((Hitters_G_played_df['MI_scale_factor'].isna()) | (Hitters_G_played_df['MI_scale_factor'] < 1))].reset_index(drop=True)

In [ ]:
UT_candidates = UT_candidates.join(UT_candidates.groupby('TeamID')['TOTAL_SGP_raw_mySGP'].rank('first',ascending=False).reset_index(name='Util_Rank'))

In [ ]:
UT_candidates['G_Util_avail'] = UT_candidates['G_mySGP'] - UT_candidates['capped_G_mySGP'] - UT_candidates['capped_MI_G_mySGP'].fillna(0)

In [ ]:
#display(UT_candidates[UT_candidates['TeamID'] == 240][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'G_MI_avail','MI_scale_factor','capped_MI_G_mySGP','Util_Rank','G_Util_avail']])

In [ ]:
def apply_limit_and_scale_Util(group):
    # Sort by rank
    group = group.sort_values('Util_Rank').reset_index()
    limit = Games_IP_Targets['Util']
    
    # Calculate cumulative sum
    group['cum_sum'] = group['G_Util_avail'].cumsum()
    
    # Identify row where limit is exceeded
    group['over_limit'] = group['cum_sum'] > limit
    
    # Initialize scale factor
    group['Util_scale_factor'] = 1.0
    
    # Find first row over limit
    over_limit_idx = group[group['over_limit']].index

    #print(group)
    
    if not over_limit_idx.empty:
        idx = over_limit_idx[0]
        # Calculate factor for the boundary row
        prev_sum = group.loc[:idx-1, 'G_Util_avail'].sum() if idx > group.index[0] else 0
        remaining = limit - prev_sum
        group.loc[idx, 'Util_scale_factor'] = max(0, remaining / group.loc[idx, 'G_Util_avail'])
        
        # Zero out rows after the limit
        group.loc[idx+1:, 'Util_scale_factor'] = 0.0
        #group.loc[idx+1:, 'G_MI_avail'] = 0.0
        # Cap the breaking row
        #group.loc[idx, 'G_MI_avail'] = max(0, remaining)

    
    group['capped_Util_G_mySGP'] = group['G_Util_avail'] * group['Util_scale_factor']
        
    return group.drop(columns=['cum_sum', 'over_limit'])

# Apply to groups
UT_candidates = UT_candidates.groupby('TeamID').apply(apply_limit_and_scale_Util).reset_index()
#print(MI_candidates)

In [ ]:
#display(UT_candidates[UT_candidates['TeamID'] == 240][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'G_MI_avail','MI_scale_factor','capped_MI_G_mySGP','Util_Rank','G_Util_avail','Util_scale_factor','capped_Util_G_mySGP']])

In [ ]:
UT_candidates = UT_candidates.drop(['level_1','level_0','index'],axis=1)

In [ ]:
Hitters_G_played_df = Hitters_G_played_df.merge(UT_candidates,how='left')

In [ ]:
display(Hitters_G_played_df[Hitters_G_played_df['TeamID'] == 240][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','G_mySGP','G_mySGP_scale_factor', 'capped_G_mySGP', 'G_MI_avail','MI_scale_factor','capped_MI_G_mySGP','Util_Rank','G_Util_avail','Util_scale_factor','capped_Util_G_mySGP']])

In [ ]:
Hitters_G_played_df['G_mySGP_Team'] = Hitters_G_played_df['capped_G_mySGP'] + Hitters_G_played_df['capped_MI_G_mySGP'].fillna(0) + Hitters_G_played_df['capped_Util_G_mySGP'].fillna(0)

In [ ]:
Hitters_G_played_df['Hitter_Team_G_mySGP_SF'] = Hitters_G_played_df['G_mySGP_Team']/Hitters_G_played_df['G_mySGP']

In [ ]:
display(Hitters_G_played_df[Hitters_G_played_df['TeamID'] == 240][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','MI_Rank','Util_Rank','G_mySGP','G_mySGP_Team','Hitter_Team_G_mySGP_SF']])

In [ ]:
def group_and_limit_sum_IP(df, limits_dict):
    """
    Groups a DataFrame by columns 'TeamID', then in order of rank on 'IP_Rank', 
    sums column 'IP_mySGP' until a limit (from limits_dict, keyed by 'Roster_Spot') is reached, 
    adding a column 'scale_factor' for the final rank to match the remainder.

    Args:
        df (pd.DataFrame): The input DataFrame with columns 'TeamID', 'IP_Rank', 'IP_mySGP'.
        limits_dict (dict): A dictionary whith limits 
                            and values are the sum limits for column 'IP_mySGP'.

    Returns:
        pd.DataFrame: The modified DataFrame with added 'cumulative_sum' and 
                      'scale_factor' columns. The 'scale_factor' is 1.0 for 
                      ranks fully included and <= 1.0 for the final, partially 
                      included rank.
    """
    
    # 1. Sort the DataFrame by columns 'TeamID', 'Roster_Spot', and 'Roster_Spot_Rank' (for ranking)
    # This ensures correct ordering within each group before applying cumulative logic.
    df_sorted = df.sort_values(by=['TeamID', 'IP_Rank']).reset_index(drop=True)

    # 2. Define a custom function to apply to each group
    def apply_limit_and_scale(group):
        b_key = 'IP'
        limit = limits_dict.get(b_key, group['IP_mySGP'].sum()) # Default to total sum if key missing
        
        # Calculate cumulative sum
        cum_sum = group['IP_mySGP'].cumsum()
        #print(b_key, cum_sum)
        
        # Determine which rows exceed the limit
        exceeded_limit = cum_sum > limit
        
        # Identify the first row that exceeds the limit
        first_exceed_index = exceeded_limit.idxmax() if exceeded_limit.any() else None
        
        # Calculate scale factors
        scale_factors = pd.Series(1.0, index=group.index)
        if first_exceed_index is not None:
            # For rows after the first exceeded, set factor to 0
            scale_factors.loc[first_exceed_index+1:] = 0.0
            
            # For the first exceeded row, calculate the proportional scale factor
            previous_sum = cum_sum.loc[:first_exceed_index-1].iloc[-1] if first_exceed_index != group.index[0] else 0 #.sum()
            remaining = limit - previous_sum
            scale_factors.loc[first_exceed_index] = remaining / group.loc[first_exceed_index, 'IP_mySGP']
            
        group['IP_mySGP_scale_factor'] = scale_factors
        group['capped_IP_mySGP'] = group['IP_mySGP'] * group['IP_mySGP_scale_factor']
        group['cumulative_capped_IP_mySGP_sum'] = group['capped_IP_mySGP'].cumsum()
        
        return group

    # 3. Apply the custom function to each group
    # We use groupby on 'TeamID' and then apply the custom function
    df_result = df_sorted.groupby(['TeamID']).apply(apply_limit_and_scale)

    return df_result.sort_index() # Revert to original index order if needed

In [ ]:
Pitcher_candidates = All_teams_replacement_level[All_teams_replacement_level['Roster_Spot'].isin(['SP','RP'])].reset_index(drop=True).join(All_teams_replacement_level[All_teams_replacement_level['Roster_Spot'].isin(['SP','RP'])].groupby('TeamID')['TOTAL_SGP_raw_mySGP'].rank('first',ascending=False).reset_index(name='IP_Rank'))

In [ ]:
Pitcher_candidates = Pitcher_candidates.drop('index',axis=1)

In [ ]:
#display(Pitcher_candidates[Pitcher_candidates['TeamID'] == 240][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','IP_Rank','IP_mySGP']])

In [ ]:
Pitchers_IP_played_df = group_and_limit_sum_IP(Pitcher_candidates, Games_IP_Targets).reset_index()

In [ ]:
Pitchers_IP_played_df = Pitchers_IP_played_df.drop('level_1',axis=1)

In [ ]:
Pitchers_IP_played_df['Pitcher_Team_IP_mySGP_SF'] = Pitchers_IP_played_df['capped_IP_mySGP']/Pitchers_IP_played_df['IP_mySGP']

In [ ]:
display(Pitchers_IP_played_df[Pitchers_IP_played_df['TeamID'] == 240][['Name','TeamID', 'Team Name','Ottoneu Positions','TOTAL_SGP_raw_mySGP','TOTAL_SGP_Val_mySGP','Roster_Spot','Roster_Spot_Rank','IP_Rank','IP_mySGP','IP_mySGP_scale_factor', 'capped_IP_mySGP', 'cumulative_capped_IP_mySGP_sum','Pitcher_Team_IP_mySGP_SF']])

In [ ]:
hitting_cols_to_scale = [
 'G_mySGP',
 'PA_mySGP',
 'AB_mySGP',
 'H_mySGP',
 '1B_mySGP',
 '2B_mySGP',
 '3B_mySGP',
 'HR_mySGP',
 'R_mySGP',
 'RBI_mySGP',
 'BB_mySGP',
 'HBP_mySGP',
 'SF_mySGP',
 'R_SGP_raw_mySGP',
 'HR_SGP_raw_mySGP',
 'OBP_SGP_raw_mySGP',
 'SLG_SGP_raw_mySGP',
 'TOTAL_SGP_raw_mySGP',
 'R_SGP_Val_mySGP',
 'HR_SGP_Val_mySGP',
 'OBP_SGP_Val_mySGP',
 'SLG_SGP_Val_mySGP',
 'PA_FGAV',
 'mR_FGAV',
 'mHR_FGAV',
 'mOBP_FGAV',
 'mSLG_FGAV',
 'PTS_FGAV',
]

pitching_cols_to_scale = [
 'W_mySGP',
 'L_mySGP',
 'QS_mySGP',
 'GS_mySGP',
 'SV_mySGP',
 'HLD_mySGP',
 'IP_mySGP',
 'H_mySGP',
 'BB_mySGP',
 'TBF_mySGP',
 'ER_mySGP',
 'HR_mySGP',
 'SO_mySGP',
 'SO_SGP_raw_mySGP',
 'HR9_SGP_raw_mySGP',
 'ERA_SGP_raw_mySGP',
 'WHIP_SGP_raw_mySGP',
 'SO_SGP_Val_mySGP',
 'HR9_SGP_Val_mySGP',
 'ERA_SGP_Val_mySGP',
 'WHIP_SGP_Val_mySGP',
 'IP_FGAV',
 'mERA_FGAV',
 'mHR_FGAV',
 'mWHIP_FGAV',
 'mSO_FGAV',
 'PTS_FGAV',
]

In [ ]:
list(Pitchers_IP_played_df.dropna(axis=1).columns)

In [ ]:
for col in hitting_cols_to_scale:
    Hitters_G_played_df[f'{col}_Team'] = Hitters_G_played_df[col]*Hitters_G_played_df['Hitter_Team_G_mySGP_SF']

for col in pitching_cols_to_scale:
    Pitchers_IP_played_df[f'{col}_Team'] = Pitchers_IP_played_df[col]*Pitchers_IP_played_df['Pitcher_Team_IP_mySGP_SF']

In [ ]:
Pitchers_IP_played_df[['Team Name','Pitcher_Team_IP_mySGP_SF']]

In [ ]:
from Stat_modules import HR9,ERA,WHIP
Pitchers_IP_played_df = Pitchers_IP_played_df.merge(
    Pitchers_IP_played_df.groupby('TeamID').apply(lambda x: HR9(
    x['HR_mySGP_Team'].sum(),
    x['IP_mySGP_Team'].sum()
)).reset_index(name='HR9_Team')
)
Pitchers_IP_played_df = Pitchers_IP_played_df.merge(
    Pitchers_IP_played_df.groupby(['TeamID','Repl_Pos_mySGP']).apply(lambda x: HR9(
    x['HR_mySGP_Team'].sum(),
    x['IP_mySGP_Team'].sum()
)).reset_index(name='HR9_pos_Team')
)

Pitchers_IP_played_df = Pitchers_IP_played_df.merge(
    Pitchers_IP_played_df.groupby('TeamID').apply(lambda x: ERA(
    x['ER_mySGP_Team'].sum(),
    x['IP_mySGP_Team'].sum()
)).reset_index(name='ERA_Team')
)

Pitchers_IP_played_df = Pitchers_IP_played_df.merge(
    Pitchers_IP_played_df.groupby(['TeamID','Repl_Pos_mySGP']).apply(lambda x: ERA(
    x['ER_mySGP_Team'].sum(),
    x['IP_mySGP_Team'].sum()
)).reset_index(name='ERA_pos_Team')
)

Pitchers_IP_played_df = Pitchers_IP_played_df.merge(
    Pitchers_IP_played_df.groupby('TeamID').apply(lambda x: WHIP(
    x['BB_mySGP_Team'].sum(),
    x['H_mySGP_Team'].sum(),
    x['IP_mySGP_Team'].sum()
)).reset_index(name='WHIP_Team')
)

Pitchers_IP_played_df = Pitchers_IP_played_df.merge(
    Pitchers_IP_played_df.groupby(['TeamID','Repl_Pos_mySGP']).apply(lambda x: WHIP(
    x['BB_mySGP_Team'].sum(),
    x['H_mySGP_Team'].sum(),
    x['IP_mySGP_Team'].sum()
)).reset_index(name='WHIP_pos_Team')
)

In [ ]:
from Stat_modules import OBP,SLG,TB
Hitters_G_played_df = Hitters_G_played_df.merge(
    Hitters_G_played_df.groupby('TeamID').apply(lambda x: OBP(
    x['H_mySGP_Team'].sum(),
    x['BB_mySGP_Team'].sum(),
    x['HBP_mySGP_Team'].sum(),
    x['SF_mySGP_Team'].sum(),
    x['AB_mySGP_Team'].sum()
)).reset_index(name='OBP_Team')
)

Hitters_G_played_df = Hitters_G_played_df.merge(
    Hitters_G_played_df.groupby(['TeamID','Repl_Pos_mySGP']).apply(lambda x: OBP(
    x['H_mySGP_Team'].sum(),
    x['BB_mySGP_Team'].sum(),
    x['HBP_mySGP_Team'].sum(),
    x['SF_mySGP_Team'].sum(),
    x['AB_mySGP_Team'].sum()
)).reset_index(name='OBP_pos_Team')
)

Hitters_G_played_df = Hitters_G_played_df.merge(
    Hitters_G_played_df.groupby('TeamID').apply(lambda x: SLG(
    TB(
        x['1B_mySGP_Team'].sum(),
        x['2B_mySGP_Team'].sum(),
        x['3B_mySGP_Team'].sum(),
        x['HR_mySGP_Team'].sum()
    ),
    x['AB_mySGP_Team'].sum()
)).reset_index(name='SLG_Team')
)

Hitters_G_played_df = Hitters_G_played_df.merge(
    Hitters_G_played_df.groupby(['TeamID','Repl_Pos_mySGP']).apply(lambda x: SLG(
    TB(
        x['1B_mySGP_Team'].sum(),
        x['2B_mySGP_Team'].sum(),
        x['3B_mySGP_Team'].sum(),
        x['HR_mySGP_Team'].sum()
    ),
    x['AB_mySGP_Team'].sum()
)).reset_index(name='SLG_pos_Team')
)

In [ ]:
All_teams_replacement_level = pd.concat([
    Pitchers_IP_played_df,
    Hitters_G_played_df,
    All_teams_replacement_level[All_teams_replacement_level['Repl_Pos_mySGP'] == 'MiLB']
])

In [ ]:
list(All_teams_replacement_level.columns)

In [ ]:
Batting_value_columns = {
    'G':['TOTAL_SGP_Val_mySGP','G_mySGP_Team','PA_mySGP_Team','PA_FGAV_Team','Salary'],
    'R':['R_mySGP_Team','R_SGP_Val_mySGP_Team'], #'R_SGP_raw_mySGP','R_SGP_norm_mySGP','mR_FGAV'
    'HR':['HR_mySGP_Team','HR_SGP_Val_mySGP_Team'], # 'HR_SGP_raw_mySGP','HR_SGP_norm_mySGP','mHR_FGAV'
    'OBP':['OBP_Team','OBP_SGP_Val_mySGP_Team'], # 'OBP_SGP_raw_mySGP','OBP_SGP_norm_mySGP','mOBP_FGAV'
    'SLG':['SLG_Team','SLG_SGP_Val_mySGP_Team'] #'SLG_SGP_raw_mySGP','SLG_SGP_norm_mySGP',,'mSLG_FGAV'
}

Pitching_value_columns = {
    'IP':['TOTAL_SGP_Val_mySGP','IP_mySGP_Team','IP_FGAV_Team','Salary'],
    'SO':['SO_mySGP_Team','SO_SGP_Val_mySGP_Team'], # 'SO_SGP_raw_mySGP','SO_SGP_norm_mySGP',,'mSO_FGAV'
    'HR9':['HR9_Team','HR9_SGP_Val_mySGP_Team'], #'HR9_SGP_raw_mySGP','HR9_SGP_norm_mySGP', ,'mHR_FGAV'
    'ERA':['ERA_Team','ERA_SGP_Val_mySGP_Team'], # 'ERA_SGP_raw_mySGP','ERA_SGP_norm_mySGP',,'mERA_FGAV'
    'WHIP':['WHIP_Team','WHIP_SGP_Val_mySGP_Team'] # 'WHIP_SGP_raw_mySGP','WHIP_SGP_norm_mySGP',,'mWHIP_FGAV'
}

In [ ]:
All_teams_replacement_level.groupby('Team Name')[[
        # 'IP_mySGP',
        # 'SO_mySGP',
        'HR9_Team',
        # 'ERA_mySGP',
        # 'WHIP_mySGP'
        ]].value_counts()

In [ ]:
Team_Rate_Cols = [
    'HR9_Team',
    'ERA_Team',
    'WHIP_Team',
    'OBP_Team',
    'SLG_Team'
]

In [ ]:
for pos in ['SP','Util']:
    This_Pos_Stat_Cats = Pitching_value_columns if pos in Pitching_Pos else Batting_value_columns
    for stat_cat, stat_cols in This_Pos_Stat_Cats.items():
        # if Stat in bugged_cols:
        #     continue
        print(pos)
        print(stat_cat)
        # print(All_teams_replacement_level[stat_cat])
        # fig1 = plt.figure(figsize=(10,5))
        # ax1 = fig1.add_subplot(111)
        # sns.histplot(All_teams_replacement_level,x='Team Name',weights=stat_cat,hue='Ottoneu Positions', multiple='stack',ax=ax1)
        # ax1.axhline(y=Target_Stats_df[stat_cat])
        # plt.tight_layout()
        # plt.show()
        
        

        for stat in stat_cols:
            print(stat)
            fig2 = plt.figure(figsize=(10,5))
            ax2 = fig2.add_subplot(111)
            if stat in Team_Rate_Cols:
                sns.histplot(All_teams_replacement_level[All_teams_replacement_level['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_teams_replacement_level)].dropna(subset=stat).groupby('Team Name')[stat].mean().reset_index(),y='Team Name',weights=stat,color='darkorange',ax=ax2)
                sns.scatterplot(All_teams_replacement_level[All_teams_replacement_level['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_teams_replacement_level)].dropna(subset=stat.replace('_','_pos_')),y='Team Name',x=stat.replace('_','_pos_'),hue='Repl_Pos_mySGP',ax=ax2)
            else:
                sns.histplot(All_teams_replacement_level[All_teams_replacement_level['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_teams_replacement_level)].dropna(subset=stat),y='Team Name',weights=stat,hue='Repl_Pos_mySGP', multiple='stack',ax=ax2)
            ax2.tick_params(axis='x', labelrotation=45)
            ax2.set_xlabel(stat)
            ax2.set_ylabel('Team Name')
            if (stat in Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')].dropna(axis=1).columns):
                ax2.axvline(x=Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')][stat].unique()[0],label='Target')
            else:
                print(Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')].dropna(axis=1).columns)
            plt.tight_layout()
            plt.show()
            #sns.histplot(All_teams_replacement_level.dropna(subset=stat),y=stat,hue=)

In [ ]:
All_teams_replacement_level

In [ ]:
All_Teams_csv_path = League_path+'/Rosters'+f'/Kept_Players_and_Replacement_Level_{Calc_date}.csv'

In [ ]:
All_teams_replacement_level.to_csv(All_Teams_csv_path,index=False)